# Continuous Sample Analytics

This notebook evaluates a trained model on `c_test`, computes per-sample losses for a configurable list of metrics (built-ins + custom callables), exports analytics CSV files, plots per-loss histograms, and visualizes representative samples from each quartile.

In [1]:
from __future__ import annotations

import json
import random
import sys
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# Ensure repository root is importable when notebook is run from elsewhere.
REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import evaluate_from_disk as efd
import NO_utilities as nou

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Repo root: {REPO_ROOT}")

Repo root: D:\Research\NO-2D-Metamaterials


In [2]:
# Configuration
MODEL_PATH = Path(
    # r"D:\Research\NO-2D-Metamaterials\MODELS\training_runs"
    # r"\NO_I3O5_BCF16_L1&L2_HC128_LR2e-03_WD0e+00_SS1_G9e-01_ch0u_260401"
    r"\NO_I3O5_BCF16_L1&L2_HC128_LR2e-03_WD0e+00_SS1_G9e-01_ch0u_260401_E30.pth"
)
OUTPUT_ROOT = Path(r"D:\Research\NO-2D-Metamaterials\DATASETS")
ANALYTICS_DIR = REPO_ROOT / "ANALYTICS"
PLOTS_DIR = REPO_ROOT / "PLOTS"
DATASET_NAME = "c_test"

ALLOW_CPU = True
BATCH_SIZE = 520
NUM_WORKERS = 1
PREFETCH_FACTOR = 2
PIN_MEMORY = True
AMP_MODE = "none"  # one of: none, fp16, bf16

# Fraction of c_test to use for faster runs. Keep at 1.0 for full evaluation.
SUBSET_FRACTION = 0.10

# Optional cap for smoke testing after any fraction-based subsampling.
DEBUG_MAX_SAMPLES = 0

ANALYTICS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model: {MODEL_PATH}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Analytics dir: {ANALYTICS_DIR}")
print(f"Plots dir: {PLOTS_DIR}")

Model: \NO_I3O5_BCF16_L1&L2_HC128_LR2e-03_WD0e+00_SS1_G9e-01_ch0u_260401_E30.pth
Output root: D:\Research\NO-2D-Metamaterials\DATASETS
Analytics dir: D:\Research\NO-2D-Metamaterials\ANALYTICS
Plots dir: D:\Research\NO-2D-Metamaterials\PLOTS


In [3]:
def _hparams_from_resolved_config(run_dir: Path) -> dict:
    rc = run_dir / "resolved_config.json"
    if not rc.is_file():
        return {}
    try:
        data = json.loads(rc.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return {}

    p = data.get("params") or {}
    a = data.get("args") or {}
    out: dict = {}
    for key in ("out_channels", "hidden_channels", "layers", "modes_height", "modes_width"):
        if key in p and p[key] is not None:
            out[key] = p[key]

    enc = a.get("eigen_ch0_encoding") or p.get("eigen_ch0_encoding")
    if isinstance(enc, str) and enc:
        out["eigen_ch0_encoding"] = enc

    amp = a.get("amp") or p.get("amp")
    if isinstance(amp, str) and amp:
        out["amp"] = amp

    return out

run_dir = MODEL_PATH.resolve().parent
hp = _hparams_from_resolved_config(run_dir)
out_channels = int(hp.get("out_channels", 5))
hidden_channels = int(hp.get("hidden_channels", 128))
layers = int(hp.get("layers", 4))
modes_height = int(hp.get("modes_height", 32))
modes_width = int(hp.get("modes_width", 32))
eigen_ch0_encoding = str(hp.get("eigen_ch0_encoding", "uniform"))
amp_mode = AMP_MODE if AMP_MODE is not None else str(hp.get("amp", "none"))

print({
    "out_channels": out_channels,
    "hidden_channels": hidden_channels,
    "layers": layers,
    "modes_height": modes_height,
    "modes_width": modes_width,
    "eigen_ch0_encoding": eigen_ch0_encoding,
    "amp_mode": amp_mode,
})

{'out_channels': 5, 'hidden_channels': 128, 'layers': 4, 'modes_height': 32, 'modes_width': 32, 'eigen_ch0_encoding': 'uniform', 'amp_mode': 'none'}


In [4]:
# Loss definitions. Each callable must return shape [B] for (pred, target).
def loss_l1_per_sample(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return (pred - target).abs().flatten(1).mean(dim=1)


def loss_l2_per_sample(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    # L2 requested in this project maps to MSE-style mean squared error.
    return (pred - target).pow(2).flatten(1).mean(dim=1)


def custom_weighted_l1(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    # Example custom metric: emphasize channel 0 by 2x.
    weights = torch.ones(pred.shape[1], device=pred.device, dtype=pred.dtype)
    weights[0] = 2.0
    abs_err = (pred - target).abs()
    weighted = abs_err * weights.view(1, -1, 1, 1)
    return weighted.flatten(1).mean(dim=1)


LOSS_REGISTRY: dict[str, Callable[[torch.Tensor, torch.Tensor], torch.Tensor]] = {
    "l1": loss_l1_per_sample,
    "l2": loss_l2_per_sample,
    "custom_weighted_l1": custom_weighted_l1,
}

# Edit this list to select which losses to evaluate.
SELECTED_LOSSES = ["l1", "l2", "custom_weighted_l1"]

missing = [name for name in SELECTED_LOSSES if name not in LOSS_REGISTRY]
if missing:
    raise ValueError(f"Missing loss definitions for: {missing}")

print(f"Selected losses: {SELECTED_LOSSES}")

Selected losses: ['l1', 'l2', 'custom_weighted_l1']


In [ ]:
all_shards = efd.discover_test_shards(OUTPUT_ROOT, eigen_ch0_encoding)
picked = [s for s in all_shards if s.name == DATASET_NAME]
if not picked:
    raise RuntimeError(f"No shard named {DATASET_NAME!r}. Found: {[s.name for s in all_shards]}")

full_dataset = efd.ShardedTensorPairDataset(picked, out_channels=out_channels)
full_indices = np.arange(len(full_dataset), dtype=np.int64)

if not (0 < float(SUBSET_FRACTION) <= 1.0):
    raise ValueError(f"SUBSET_FRACTION must be in (0, 1]. Got {SUBSET_FRACTION}")

if float(SUBSET_FRACTION) < 1.0:
    rng = np.random.default_rng(SEED)
    subset_n = max(1, int(np.floor(len(full_dataset) * float(SUBSET_FRACTION))))
    subset_indices = np.sort(rng.choice(full_indices, size=subset_n, replace=False))
else:
    subset_indices = full_indices

if DEBUG_MAX_SAMPLES and DEBUG_MAX_SAMPLES > 0:
    subset_indices = subset_indices[: min(int(DEBUG_MAX_SAMPLES), len(subset_indices))]

dataset = torch.utils.data.Subset(full_dataset, subset_indices.tolist())
dataset_indices = subset_indices.copy()

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "shuffle": False,
    "num_workers": NUM_WORKERS,
    "pin_memory": bool(PIN_MEMORY),
    "drop_last": False,
}
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

loader = DataLoader(dataset, **loader_kwargs)

device = efd.resolve_device(ALLOW_CPU)
model = efd.FourierNeuralOperator(
    modes_height=modes_height,
    modes_width=modes_width,
    hidden_channels=hidden_channels,
    n_layers=layers,
    out_channels=out_channels,
).to(device)
efd.load_model_state(model, MODEL_PATH.resolve())
model.eval()

print(f"Dataset: {DATASET_NAME}")
print(f"SUBSET_FRACTION: {SUBSET_FRACTION}")
print(f"Samples to evaluate: {len(dataset)} / {len(full_dataset)}")
print(f"Device: {device}")

In [ ]:
# Run inference once and compute every selected loss per sample.
loss_buffers = {name: torch.empty(len(dataset), dtype=torch.float64) for name in SELECTED_LOSSES}

cursor = 0
model.eval()
with torch.inference_mode():
    for xb, yb in loader:
        xb = xb.to(device, dtype=torch.float32, non_blocking=True)
        yb = yb.to(device, dtype=torch.float32, non_blocking=True)
        with efd.amp_context(device, amp_mode):
            pred = model(xb)

        batch_size = int(xb.shape[0])
        for loss_name in SELECTED_LOSSES:
            per_sample = LOSS_REGISTRY[loss_name](pred, yb)
            if per_sample.shape != (batch_size,):
                raise ValueError(
                    f"Loss {loss_name!r} must return shape [B]. Got {tuple(per_sample.shape)}"
                )
            loss_buffers[loss_name][cursor : cursor + batch_size] = (
                per_sample.detach().to(torch.float64).cpu()
            )

        cursor += batch_size

if cursor != len(dataset):
    raise RuntimeError(f"Sample count mismatch: wrote {cursor}, expected {len(dataset)}")

print(f"Computed losses for {cursor} samples and losses={SELECTED_LOSSES}")

In [ ]:
# Build and save the primary per-sample CSV.
records = {
    "sample_index": dataset_indices,
}
for loss_name in SELECTED_LOSSES:
    records[loss_name] = loss_buffers[loss_name].numpy()

df_losses = pd.DataFrame(records).sort_values("sample_index").reset_index(drop=True)

run_tag = MODEL_PATH.stem
per_sample_csv = ANALYTICS_DIR / f"{DATASET_NAME}_{run_tag}_per_sample_losses.csv"
df_losses.to_csv(per_sample_csv, index=False)

summary_rows = []
for loss_name in SELECTED_LOSSES:
    vals = df_losses[loss_name].to_numpy()
    summary_rows.append(
        {
            "loss": loss_name,
            "mean": float(np.mean(vals)),
            "median": float(np.median(vals)),
            "std": float(np.std(vals)),
            "min": float(np.min(vals)),
            "max": float(np.max(vals)),
        }
    )

df_summary = pd.DataFrame(summary_rows)
summary_csv = ANALYTICS_DIR / f"{DATASET_NAME}_{run_tag}_loss_summary.csv"
df_summary.to_csv(summary_csv, index=False)

print(f"Saved per-sample loss CSV: {per_sample_csv}")
print(f"Saved summary CSV: {summary_csv}")
df_summary

In [ ]:
# Quartile extraction for each loss.
quartile_rows = []
representative_rows = []
quartile_labels = ["Q1", "Q2", "Q3", "Q4"]

for loss_name in SELECTED_LOSSES:
    work = df_losses[["sample_index", loss_name]].copy()
    work = work.sort_values([loss_name, "sample_index"], kind="mergesort").reset_index(drop=True)

    q25, q50, q75 = work[loss_name].quantile([0.25, 0.5, 0.75]).tolist()
    boundaries = [(-np.inf, q25), (q25, q50), (q50, q75), (q75, np.inf)]

    for qlabel, (lo, hi) in zip(quartile_labels, boundaries):
        if qlabel == "Q1":
            bucket = work[work[loss_name] <= hi]
        elif qlabel == "Q2":
            bucket = work[(work[loss_name] > lo) & (work[loss_name] <= hi)]
        elif qlabel == "Q3":
            bucket = work[(work[loss_name] > lo) & (work[loss_name] <= hi)]
        else:
            bucket = work[work[loss_name] > lo]

        if bucket.empty:
            continue

        bucket = bucket.copy()
        bucket["loss"] = loss_name
        bucket["quartile"] = qlabel
        quartile_rows.append(bucket[["loss", "quartile", "sample_index", loss_name]].rename(columns={loss_name: "loss_value"}))

        q_median = float(bucket[loss_name].median())
        rep = bucket.iloc[(bucket[loss_name] - q_median).abs().argmin()]
        representative_rows.append(
            {
                "loss": loss_name,
                "quartile": qlabel,
                "sample_index": int(rep["sample_index"]),
                "loss_value": float(rep[loss_name]),
                "quartile_median": q_median,
            }
        )

quartile_df = pd.concat(quartile_rows, ignore_index=True)
quartile_csv = ANALYTICS_DIR / f"{DATASET_NAME}_{run_tag}_quartile_membership.csv"
quartile_df.to_csv(quartile_csv, index=False)

representatives_df = pd.DataFrame(representative_rows).sort_values(["loss", "quartile"])
representatives_csv = ANALYTICS_DIR / f"{DATASET_NAME}_{run_tag}_quartile_representatives.csv"
representatives_df.to_csv(representatives_csv, index=False)

print(f"Saved quartile membership CSV: {quartile_csv}")
print(f"Saved quartile representatives CSV: {representatives_csv}")
representatives_df

In [ ]:
# Histogram per selected loss.
histogram_paths = []
for loss_name in SELECTED_LOSSES:
    vals = df_losses[loss_name].to_numpy()
    out_path = PLOTS_DIR / f"{DATASET_NAME}_{run_tag}_{loss_name}_per_sample_histogram.png"

    fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax.hist(vals, bins=80, color="steelblue", edgecolor="white", linewidth=0.3)
    ax.set_xlabel(f"Per-sample {loss_name} loss")
    ax.set_ylabel("Count")
    ax.set_title(f"{DATASET_NAME} - {loss_name} ({MODEL_PATH.name})")
    ax.axvline(float(vals.mean()), color="darkorange", linestyle="--", linewidth=1.5, label=f"mean = {float(vals.mean()):.4g}")
    ax.axvline(float(np.median(vals)), color="darkgreen", linestyle=":", linewidth=1.5, label=f"median = {float(np.median(vals)):.4g}")
    ax.legend()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

    histogram_paths.append(out_path)
    print(f"Saved histogram: {out_path}")

In [ ]:
# Quartile visualization using NO_utilities.visualize_sample.
# This section renders one representative sample for each quartile of each loss.

def predict_single(index: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    x, y = full_dataset[int(index)]
    with torch.inference_mode():
        xb = x.unsqueeze(0).to(device=device, dtype=torch.float32)
        with efd.amp_context(device, amp_mode):
            pred = model(xb).squeeze(0).detach().cpu().to(torch.float32)
    return x.detach().cpu().to(torch.float32), y.detach().cpu().to(torch.float32), pred

for loss_name in SELECTED_LOSSES:
    reps = representatives_df[representatives_df["loss"] == loss_name].copy()
    if reps.empty:
        print(f"No representatives found for loss={loss_name}")
        continue

    print("=" * 90)
    print(f"Visualizing quartile representatives for loss={loss_name}")
    for qlabel in quartile_labels:
        row = reps[reps["quartile"] == qlabel]
        if row.empty:
            continue
        row = row.iloc[0]
        idx = int(row["sample_index"])
        lv = float(row["loss_value"])
        print(f"{loss_name} | {qlabel} | sample_index={idx} | loss_value={lv:.6e}")

        x, y, yhat = predict_single(idx)
        nou.visualize_sample(
            input_tensor=x,
            output_tensor=yhat,
            target_tensor=y,
            diffs=True,
            unified_colorbar=True,
            field_cmap="RdBu_r",
            diverge_center=0.0,
        )

In [ ]:
print("Run complete.")
print(f"Per-sample CSV: {per_sample_csv}")
print(f"Summary CSV: {summary_csv}")
print(f"Quartile membership CSV: {quartile_csv}")
print(f"Quartile representatives CSV: {representatives_csv}")
print("Histogram PNGs:")
for p in histogram_paths:
    print(f"  - {p}")